In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! mkdir ~/.kaggle

In [ ]:
! cp /content/drive/MyDrive/"Colab Notebooks"/kaggle.json ~/.kaggle/

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
! kaggle datasets download -d ashfakyeafi/spam-email-classification

Dataset URL: https://www.kaggle.com/datasets/ashfakyeafi/spam-email-classification
License(s): apache-2.0
  0% 0.00/207k [00:00<?, ?B/s]
100% 207k/207k [00:00<00:00, 478MB/s]


In [ ]:
! unzip spam-email-classification.zip

Archive:  spam-email-classification.zip
  inflating: email.csv               


In [ ]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [ ]:
df = pd.read_csv('email.csv')

In [ ]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [ ]:
df.describe()

,Category,Message
count,5573,5573
unique,3,5158
top,ham,"Sorry, I'll call later"
freq,4825,30


In [ ]:
df['Category'].value_counts()

,count
Category,
ham,4825
spam,747
"{""mode"":""full""",1


In [ ]:
import string

#preprocessing the data

def preprocess_text(text):
  text = text.lower()
  text = text.translate(str.maketrans('', '', string.punctuation))
  return text

In [ ]:
df['Message'] = df['Message'].apply(preprocess_text)

In [ ]:
#tokenize

def tokenize(text):
  return text.split()

df['tokens'] = df['Message'].apply(tokenize)

In [ ]:
df.head()

,Category,Message,tokens
0,ham,go until jurong point crazy available only in ...,"[go, until, jurong, point, crazy, available, o..."
1,ham,ok lar joking wif u oni,"[ok, lar, joking, wif, u, oni]"
2,spam,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,ham,u dun say so early hor u c already then say,"[u, dun, say, so, early, hor, u, c, already, t..."
4,ham,nah i dont think he goes to usf he lives aroun...,"[nah, i, dont, think, he, goes, to, usf, he, l..."


In [ ]:
vocab = set()

for tokens in df['tokens']:
  vocab.update(tokens)

In [ ]:
vocab

{'62220cncl',
 'asshole',
 'parked',
 'detail',
 'expecting',
 'lesson',
 'resubmit',
 '28',
 '4wrd',
 'callingforgot',
 'iam',
 'checkup',
 'panalambut',
 'wld',
 'smsd',
 'nevering',
 'fineinshah',
 'reason',
 'hopeing',
 'renting',
 'pretty',
 'attend',
 'guesses',
 'zeros',
 'dreading',
 'enna',
 'smsing',
 'dimension',
 '95pax',
 'thatmum',
 'adjustable',
 'meneed',
 'smoking',
 'pongal',
 'black',
 '2morrow',
 'regretted',
 'cartoon',
 'hostile',
 'organise',
 'isn\x92t',
 'surprise',
 'melle',
 'whn',
 '08714712388',
 'justbeen',
 'smash',
 'groovying',
 'yoga',
 'yijue',
 'crushes',
 'bro',
 'wee',
 'phil',
 'youve',
 'listening',
 'premaricakindly',
 'malarky',
 'weaknesses',
 'petrol',
 'kano',
 'kicchu',
 'callers',
 'listening2the',
 'moves',
 'spent',
 'glad',
 'forwarding',
 'dumb',
 'call2optout4qf2',
 'carry',
 'greece',
 'hen',
 '285',
 'prepared',
 'wwwidewcom',
 '81303',
 'doubles',
 'loveable',
 '£150wk',
 'ettans',
 'jason',
 'girls',
 '1childish',
 '08719899230',


In [ ]:
#initialize dictionaries

word_counts = defaultdict(lambda: {'ham':0, 'spam':0})
class_counts = {'ham':0, 'spam':0}

In [ ]:
for i, row in df.iterrows():
  class_label = row['Category']
  if class_label in class_counts:
    class_counts[class_label] += 1
    for token in row['tokens']:
      word_counts[token][class_label] += 1

  else:
    print(f"Invalid class label: {class_label}")

Invalid class label: {"mode":"full"


In [ ]:
#calculate probabilities

total_sample = len(df)
prior_ham = class_counts['ham'] / total_sample
prior_spam = class_counts['spam'] / total_sample

In [ ]:
#calculate prior probabilities

total_samples = len(df)
prior_ham = class_counts['ham'] / total_samples
prior_spam = class_counts['spam'] / total_samples

In [ ]:
#calculate likelihood probabilities

likelihood_probs = {}

for word in vocab:
  likelihood_probs[word] = {
      'ham': (word_counts[word]['ham'] + 1) / (class_counts['ham'] + len(vocab)),
      'spam': (word_counts[word]['spam'] + 1) / (class_counts['spam'] + len(vocab))
  }

In [ ]:
#prediction function

def predict(Message):
  tokens = tokenize(preprocess_text(Message))
  ham_score = np.log(prior_ham) + sum(np.log(likelihood_probs.get(word, {'ham': 1})['ham']) for word in tokens)
  spam_score = np.log(prior_spam) + sum(np.log(likelihood_probs.get(word, {'spam': 1})['spam']) for word in tokens)

  print(ham_score)
  print(spam_score)

  if ham_score > spam_score:
    return 'ham'
  else:
    return 'spam'


In [ ]:
new_email = 'Congratulations, you win this car! Click now to this link.'

predicted_class = predict(new_email)
print(f'Predicted class: {predicted_class}')

-48.65842255954074
-48.428533714136194
Predicted class: spam
